In [1]:
"""
大盘周报生成器 - 多盘合并版
每个盘一个 Sheet + 汇总对比 Sheet

★ 口径说明（与PDF周报统一）：
  充提差率 = sum(充提差) / sum(充值金额) × 100%
  盈亏率   = sum(公司输赢) / sum(投注金额) × 100%
  日均消耗 / ROI：本周固定剔除最后一天（消耗未录完），上周取全7天
"""
import warnings
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

warnings.filterwarnings("ignore")

# ══════════════════════════════════════════════════════════════════════════════
# ★★★ 配置区（每次只改这里）★★★
# ══════════════════════════════════════════════════════════════════════════════

THIS_WEEK = ("20260605", "20260611")
LAST_WEEK = ("20260529", "20260604")

# 每个盘：(sheet名称, 平台报表路径, 日报路径)
PLATFORMS = [
    (
        "MX",
        r"D:\周报更新版\MX\大盘\平台报表_USD_20260612152421.xlsx",
        r"D:\周报更新版\MX\大盘\日报-大盘日报_20260612.xlsx",
    ),
    (
        "SOLUNO",
        r"D:\周报更新版\SOLUNO\平台报表_USD_20260612170544.xlsx",
        r"D:\周报更新版\SOLUNO\日报-大盘日报_20260612(1).xlsx",
    ),
    (
        "SOL777",
        r"D:\周报更新版\SOL777\平台报表_USD_20260612170747.xlsx",
        r"D:\周报更新版\SOL777\日报-大盘日报_20260612(3).xlsx",
    ),
    (
        "Lucro",
        r"D:\周报更新版\Lucro\大盘\平台报表_USD_20260612171058.xlsx",
        r"D:\周报更新版\Lucro\大盘\日报-大盘日报_20260612(2).xlsx",
    ),
    (
        "wey7",
        r"D:\周报更新版\wey7\平台报表_USD_20260612171347.xlsx",
        r"D:\周报更新版\wey7\日报-大盘日报_20260612(4).xlsx",
    ),
    (
        "oro7x",
        r"D:\周报更新版\oro7x\平台报表_USD_20260612171448.xlsx",
        r"D:\周报更新版\oro7x\日报-大盘日报_20260612(6).xlsx",
    ),
    # 新增盘时在此追加一行：
    # ("盘名", r"平台报表路径", r"日报路径"),
]

OUTPUT_PATH = r"D:\周报更新版\大盘周报汇总.xlsx"

# ══════════════════════════════════════════════════════════════════════════════
# 颜色常量
# ══════════════════════════════════════════════════════════════════════════════
HDR_BG   = "1E3A5F"
HDR_FG   = "FFFFFF"
MET_BG   = "D6E4F0"
MET_FG   = "1E3A5F"
SUB_HDR  = "2C5F8A"
UP_FG    = "1A7C3E"
DN_FG    = "C0392B"
NEU_FG   = "888888"
ZEBRA    = "F0F6FC"
WHITE    = "FFFFFF"
BORDER_C = "B0C4D8"
COST_BG  = "FFF9E6"

# ══════════════════════════════════════════════════════════════════════════════
# 工具函数
# ══════════════════════════════════════════════════════════════════════════════
def clean(v):
    try:
        return float(str(v).replace(",", "").replace("%", "").strip())
    except Exception:
        return np.nan


def load_and_clean(path):
    df = pd.read_excel(path)
    df["日期"] = df["日期"].astype(str).str.strip()
    for col in df.columns:
        if col != "日期":
            df[col] = df[col].apply(clean)
    return df


def week_slice(df, week):
    return (
        df[df["日期"].between(week[0], week[1])]
        .sort_values("日期")
        .reset_index(drop=True)
    )


def pct_chg(tw_val, lw_val):
    if (lw_val is not None and not np.isnan(lw_val) and lw_val != 0
            and tw_val is not None and not np.isnan(tw_val)):
        return (tw_val - lw_val) / abs(lw_val) * 100
    return np.nan


def safe_sum(series):
    v = series.dropna()
    return float(v.sum()) if len(v) else np.nan


def safe_mean(series):
    v = series.dropna()
    return float(v.mean()) if len(v) else np.nan


def fmt_week_label(week):
    s, e = week
    return f"{s[4:6]}/{s[6:]}—{e[4:6]}/{e[6:]}"


# ══════════════════════════════════════════════════════════════════════════════
# 核心计算
# ══════════════════════════════════════════════════════════════════════════════
def compute_metrics(plat_path, daily_path):
    try:
        p = load_and_clean(plat_path)
        d = load_and_clean(daily_path)
    except Exception as ex:
        return None, f"文件读取失败：{ex}"

    tw_p = week_slice(p, THIS_WEEK)
    lw_p = week_slice(p, LAST_WEEK)
    tw_d = week_slice(d, THIS_WEEK)
    lw_d = week_slice(d, LAST_WEEK)

    if len(tw_p) == 0 or len(lw_p) == 0:
        return None, "数据不足（本周或上周平台报表无记录）"
    if len(tw_d) == 0 or len(lw_d) == 0:
        return None, "数据不足（本周或上周日报无记录）"

    # ── 消耗 / ROI：固定剔除本周最后一天 ────────────────────────────────────
    tw_cost_rows = tw_d.iloc[:-1]
    lw_cost_rows = lw_d
    tw_cost_days = len(tw_cost_rows)
    lw_cost_days = len(lw_cost_rows)

    tw_cost_avg = (safe_sum(tw_cost_rows["真实消耗"]) / tw_cost_days
                   if tw_cost_days else np.nan)
    lw_cost_avg = (safe_sum(lw_cost_rows["真实消耗"]) / lw_cost_days
                   if lw_cost_days else np.nan)

    # 充提差与消耗天数对齐（同样剔最后一天）
    cd_col = "充提差" if "充提差" in tw_d.columns else None
    if cd_col:
        tw_cd_valid = tw_d.iloc[:-1]["充提差"]
        lw_cd_all   = lw_d["充提差"]
    else:
        tw_cd_valid = tw_p.sort_values("日期").iloc[:-1]["充提差"]
        lw_cd_all   = lw_p.sort_values("日期")["充提差"]

    tw_cd_avg = safe_sum(tw_cd_valid) / tw_cost_days if tw_cost_days else np.nan
    lw_cd_avg = safe_sum(lw_cd_all)   / lw_cost_days if lw_cost_days else np.nan

    tw_roi = (tw_cd_avg / tw_cost_avg
              if tw_cost_avg and not np.isnan(tw_cost_avg) else np.nan)
    lw_roi = (lw_cd_avg / lw_cost_avg
              if lw_cost_avg and not np.isnan(lw_cost_avg) else np.nan)

    # ── 基础聚合 ─────────────────────────────────────────────────────────────
    tw_充值  = safe_sum(tw_p["充值金额"])
    lw_充值  = safe_sum(lw_p["充值金额"])
    tw_充提差 = safe_sum(tw_p["充提差"])
    lw_充提差 = safe_sum(lw_p["充提差"])
    tw_投注  = safe_sum(tw_p["投注金额"]) if "投注金额" in tw_p.columns else np.nan
    lw_投注  = safe_sum(lw_p["投注金额"]) if "投注金额" in lw_p.columns else np.nan
    tw_输赢  = safe_sum(tw_p["公司输赢"])
    lw_输赢  = safe_sum(lw_p["公司输赢"])
    tw_注册  = safe_sum(tw_p["注册人数"])
    lw_注册  = safe_sum(lw_p["注册人数"])
    tw_首充  = safe_sum(tw_p["首充人数"])
    lw_首充  = safe_sum(lw_p["首充人数"])
    tw_活跃  = safe_sum(tw_p["活跃人数"])
    lw_活跃  = safe_sum(lw_p["活跃人数"])

    # ── ★ 充提差率 = sum(充提差) / sum(充值金额) ────────────────────────────
    tw_充提差率 = tw_充提差 / tw_充值 * 100 if tw_充值 else np.nan
    lw_充提差率 = lw_充提差 / lw_充值 * 100 if lw_充值 else np.nan

    # ── ★ 盈亏率 = sum(公司输赢) / sum(投注金额) ────────────────────────────
    tw_盈亏率 = tw_输赢 / tw_投注 * 100 if (tw_投注 and not np.isnan(tw_投注)) else np.nan
    lw_盈亏率 = lw_输赢 / lw_投注 * 100 if (lw_投注 and not np.isnan(lw_投注)) else np.nan

    tw_日均活跃   = tw_活跃 / 7 if tw_活跃 else np.nan
    lw_日均活跃   = lw_活跃 / 7 if lw_活跃 else np.nan
    tw_首充转化率 = tw_首充 / tw_注册 * 100 if tw_注册 else np.nan
    lw_首充转化率 = lw_首充 / lw_注册 * 100 if lw_注册 else np.nan
    tw_首充ARPPU  = safe_mean(tw_p["首充Arppu"])
    lw_首充ARPPU  = safe_mean(lw_p["首充Arppu"])
    tw_老用户ARPPU = safe_mean(tw_p["老用户ARPPU"])
    lw_老用户ARPPU = safe_mean(lw_p["老用户ARPPU"])
    tw_全量ARPPU  = safe_mean(tw_p["全量Arppu"])
    lw_全量ARPPU  = safe_mean(lw_p["全量Arppu"])

    def row(label, tw_val, lw_val, fmt, note="", is_cost=False):
        return {
            "指标":    label,
            "上周":    lw_val,
            "本周":    tw_val,
            "环比":    pct_chg(tw_val, lw_val),
            "fmt":     fmt,
            "note":    note,
            "is_cost": is_cost,
        }

    cost_note = f"本周取前{tw_cost_days}天/上周取{lw_cost_days}天日均"

    rows = [
        row("充值金额（万）",          tw_充值  / 10000, lw_充值  / 10000, "万"),
        row("充提差率",                tw_充提差率,       lw_充提差率,       "%",  "sum(充提差)/sum(充值金额)"),
        row("公司输赢（万）",          tw_输赢  / 10000, lw_输赢  / 10000, "万"),
        row("盈亏率",                  tw_盈亏率,         lw_盈亏率,         "%",  "sum(公司输赢)/sum(投注金额)"),
        row("日均消耗（万）",          tw_cost_avg / 10000, lw_cost_avg / 10000, "万",  cost_note, True),
        row("日均ROI（充提差/消耗）",  tw_roi,            lw_roi,            "x",  cost_note, True),
        row("注册人数",                tw_注册,           lw_注册,           "整数"),
        row("首充人数",                tw_首充,           lw_首充,           "整数"),
        row("日均活跃",                tw_日均活跃,       lw_日均活跃,       "整数"),
        row("首充转化率",              tw_首充转化率,     lw_首充转化率,     "%"),
        row("首充ARPPU",               tw_首充ARPPU,      lw_首充ARPPU,      "小数"),
        row("老用户ARPPU",             tw_老用户ARPPU,    lw_老用户ARPPU,    "小数"),
        row("全量ARPPU",               tw_全量ARPPU,      lw_全量ARPPU,      "小数"),
    ]
    return rows, None


# ══════════════════════════════════════════════════════════════════════════════
# Excel 样式工具
# ══════════════════════════════════════════════════════════════════════════════
def mk_border(color=BORDER_C, style="thin"):
    s = Side(style=style, color=color)
    return Border(left=s, right=s, top=s, bottom=s)


def mk_fill(hex_color):
    return PatternFill("solid", fgColor=hex_color)


def fmt_val(v, fmt):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return "-"
    if fmt == "万":   return f"{v:,.2f}万"
    if fmt == "%":    return f"{v:.2f}%"
    if fmt == "x":    return f"{v:.4f}x"
    if fmt == "整数": return f"{int(round(v)):,}"
    if fmt == "小数": return f"{v:.2f}"
    return str(v)


def fmt_chg(chg):
    if chg is None or (isinstance(chg, float) and np.isnan(chg)):
        return "-"
    sign = "+" if chg >= 0 else ""
    return f"{sign}{chg:.2f}%"


# ══════════════════════════════════════════════════════════════════════════════
# 单盘 Sheet 写入
# ══════════════════════════════════════════════════════════════════════════════
def write_platform_sheet(ws, platform_name, rows):
    tw_lbl = fmt_week_label(THIS_WEEK)
    lw_lbl = fmt_week_label(LAST_WEEK)

    for col, w in zip("ABCDE", [26, 18, 18, 13, 34]):
        ws.column_dimensions[col].width = w

    # 第1行：标题
    ws.merge_cells("A1:E1")
    c = ws["A1"]
    c.value     = f"【{platform_name}】大盘周报    本周 {tw_lbl}  vs  上周 {lw_lbl}"
    c.font      = Font(name="Arial", bold=True, size=12, color=HDR_FG)
    c.fill      = mk_fill(HDR_BG)
    c.alignment = Alignment(horizontal="center", vertical="center")
    c.border    = mk_border(HDR_BG)
    ws.row_dimensions[1].height = 26

    # 第2行：表头
    headers = ["指标", f"上周（{lw_lbl}）", f"本周（{tw_lbl}）", "环比", "口径说明"]
    for col_i, hdr in enumerate(headers, 1):
        c = ws.cell(row=2, column=col_i, value=hdr)
        c.font      = Font(name="Arial", bold=True, size=10, color=HDR_FG)
        c.fill      = mk_fill(SUB_HDR)
        c.alignment = Alignment(horizontal="center", vertical="center")
        c.border    = mk_border(SUB_HDR)
    ws.row_dimensions[2].height = 20

    # 第3行起：数据
    for i, r in enumerate(rows):
        row_num = i + 3
        bg = COST_BG if r["is_cost"] else (WHITE if i % 2 == 0 else ZEBRA)

        chg     = r["环比"]
        chg_txt = fmt_chg(chg)
        chg_clr = NEU_FG if chg_txt == "-" else (UP_FG if chg >= 0 else DN_FG)

        cells = [
            (1, r["指标"],                    MET_BG, "left",
             Font(name="Arial", bold=True, size=10, color=MET_FG)),
            (2, fmt_val(r["上周"], r["fmt"]), bg,     "right",
             Font(name="Arial", size=10, color="555555")),
            (3, fmt_val(r["本周"], r["fmt"]), bg,     "right",
             Font(name="Arial", bold=True, size=10, color=MET_FG)),
            (4, chg_txt,                      bg,     "center",
             Font(name="Arial", bold=True, size=10, color=chg_clr)),
            (5, r.get("note", ""),            bg,     "left",
             Font(name="Arial", size=9, color="888888", italic=True)),
        ]
        for col_i, val, fill_c, align, font in cells:
            c = ws.cell(row=row_num, column=col_i, value=val)
            c.font      = font
            c.fill      = mk_fill(fill_c)
            c.alignment = Alignment(horizontal=align, vertical="center",
                                    indent=(1 if align == "left" else 0))
            c.border    = mk_border()
        ws.row_dimensions[row_num].height = 20

    # 底部时间戳
    ts_row = len(rows) + 3
    c = ws.cell(row=ts_row, column=1,
                value=f"生成时间：{datetime.now().strftime('%Y-%m-%d %H:%M')}")
    c.font = Font(name="Arial", size=8, color="AAAAAA", italic=True)


# ══════════════════════════════════════════════════════════════════════════════
# 汇总对比 Sheet 写入
# ══════════════════════════════════════════════════════════════════════════════
def write_summary_sheet(ws, summary):
    valid   = [(n, r) for n, r in summary if r is not None]
    n_plat  = len(valid)
    tw_lbl  = fmt_week_label(THIS_WEEK)
    lw_lbl  = fmt_week_label(LAST_WEEK)

    ws.column_dimensions["A"].width = 26
    for i in range(n_plat * 2):
        ws.column_dimensions[get_column_letter(i + 2)].width = 13

    end_col = get_column_letter(n_plat * 2 + 1)

    # 第1行：总标题
    ws.merge_cells(f"A1:{end_col}1")
    c = ws["A1"]
    c.value     = f"各盘大盘周报汇总对比    本周 {tw_lbl}  vs  上周 {lw_lbl}"
    c.font      = Font(name="Arial", bold=True, size=12, color=HDR_FG)
    c.fill      = mk_fill(HDR_BG)
    c.alignment = Alignment(horizontal="center", vertical="center")
    c.border    = mk_border(HDR_BG)
    ws.row_dimensions[1].height = 26

    # 第2行：盘名（跨两列）
    ws.cell(row=2, column=1, value="指标")
    ws["A2"].font      = Font(name="Arial", bold=True, size=10, color=HDR_FG)
    ws["A2"].fill      = mk_fill(SUB_HDR)
    ws["A2"].alignment = Alignment(horizontal="center", vertical="center")
    ws["A2"].border    = mk_border(SUB_HDR)

    for j, (name, _) in enumerate(valid):
        cs = j * 2 + 2; ce = cs + 1
        ws.merge_cells(start_row=2, start_column=cs, end_row=2, end_column=ce)
        c = ws.cell(row=2, column=cs, value=name)
        c.font      = Font(name="Arial", bold=True, size=10, color=HDR_FG)
        c.fill      = mk_fill(SUB_HDR)
        c.alignment = Alignment(horizontal="center", vertical="center")
        c.border    = mk_border(SUB_HDR)
    ws.row_dimensions[2].height = 20

    # 第3行：子表头
    ws.cell(row=3, column=1, value="").fill   = mk_fill(SUB_HDR)
    ws.cell(row=3, column=1).border           = mk_border(SUB_HDR)
    for j in range(n_plat):
        cs = j * 2 + 2
        for offset, sub in enumerate(["本周值", "环比"]):
            c = ws.cell(row=3, column=cs + offset, value=sub)
            c.font      = Font(name="Arial", bold=True, size=9, color=HDR_FG)
            c.fill      = mk_fill("3A7DBF")
            c.alignment = Alignment(horizontal="center", vertical="center")
            c.border    = mk_border("3A7DBF")
    ws.row_dimensions[3].height = 18

    # 第4行起：数据
    first_rows = valid[0][1]
    for i, tmpl in enumerate(first_rows):
        row_num = i + 4
        is_cost = tmpl["is_cost"]
        bg      = COST_BG if is_cost else (WHITE if i % 2 == 0 else ZEBRA)

        # 指标名列（含口径说明换行）
        label = tmpl["指标"]
        note  = tmpl.get("note", "")
        display = f"{label}\n({note})" if note and not is_cost else label
        c = ws.cell(row=row_num, column=1, value=display)
        c.font      = Font(name="Arial", bold=True, size=9, color=MET_FG)
        c.fill      = mk_fill(MET_BG)
        c.alignment = Alignment(horizontal="left", vertical="center",
                                indent=1, wrap_text=bool(note and not is_cost))
        c.border    = mk_border()
        if note and not is_cost:
            ws.row_dimensions[row_num].height = 30
        else:
            ws.row_dimensions[row_num].height = 20

        for j, (_, plat_rows) in enumerate(valid):
            cs = j * 2 + 2
            if plat_rows is None:
                for offset in range(2):
                    c = ws.cell(row=row_num, column=cs + offset, value="-")
                    c.fill      = mk_fill(bg)
                    c.alignment = Alignment(horizontal="center", vertical="center")
                    c.border    = mk_border()
                continue

            r       = plat_rows[i]
            val_str = fmt_val(r["本周"], r["fmt"])
            chg     = r["环比"]
            chg_txt = fmt_chg(chg)
            chg_clr = NEU_FG if chg_txt == "-" else (UP_FG if chg >= 0 else DN_FG)

            cv = ws.cell(row=row_num, column=cs, value=val_str)
            cv.font      = Font(name="Arial", size=10, color=MET_FG)
            cv.fill      = mk_fill(bg)
            cv.alignment = Alignment(horizontal="right", vertical="center")
            cv.border    = mk_border()

            cc = ws.cell(row=row_num, column=cs + 1, value=chg_txt)
            cc.font      = Font(name="Arial", bold=True, size=10, color=chg_clr)
            cc.fill      = mk_fill(bg)
            cc.alignment = Alignment(horizontal="center", vertical="center")
            cc.border    = mk_border()

    # 底部口径说明
    note_row = len(first_rows) + 5
    ws.merge_cells(f"A{note_row}:{end_col}{note_row}")
    c = ws.cell(row=note_row, column=1,
                value="口径说明：充提差率=sum(充提差)/sum(充值金额)；盈亏率=sum(公司输赢)/sum(投注金额)；"
                      "日均消耗/ROI本周取前N-1天（剔最后一天消耗未录完），上周取全7天")
    c.font      = Font(name="Arial", size=8, color="666666", italic=True)
    c.alignment = Alignment(horizontal="left", vertical="center", indent=1)

    ts_row = note_row + 1
    ws.cell(row=ts_row, column=1,
            value=f"生成时间：{datetime.now().strftime('%Y-%m-%d %H:%M')}").font = \
        Font(name="Arial", size=8, color="AAAAAA", italic=True)


# ══════════════════════════════════════════════════════════════════════════════
# 主流程
# ══════════════════════════════════════════════════════════════════════════════
def main():
    wb = openpyxl.Workbook()
    wb.remove(wb.active)

    summary = []

    for name, plat_path, daily_path in PLATFORMS:
        print(f"  处理 {name} ...")
        rows, err = compute_metrics(plat_path, daily_path)
        if err:
            print(f"    ⚠️  {name} 加载失败：{err}")
            summary.append((name, None))
            continue
        ws = wb.create_sheet(title=name)
        write_platform_sheet(ws, name, rows)
        summary.append((name, rows))
        print(f"    ✅ {name} 写入完成")

    valid_count = sum(1 for _, r in summary if r is not None)
    if valid_count >= 1:
        ws_sum = wb.create_sheet(title="汇总对比", index=0)
        write_summary_sheet(ws_sum, summary)
        print("  ✅ 汇总对比 Sheet 写入完成")

    out = Path(OUTPUT_PATH)
    out.parent.mkdir(parents=True, exist_ok=True)
    wb.save(str(out))
    print(f"\n✅ 文件已保存：{out}")


if __name__ == "__main__":
    main()

  处理 MX ...
    ✅ MX 写入完成
  处理 SOLUNO ...
    ✅ SOLUNO 写入完成
  处理 SOL777 ...
    ✅ SOL777 写入完成
  处理 Lucro ...
    ✅ Lucro 写入完成
  处理 wey7 ...
    ✅ wey7 写入完成
  处理 oro7x ...
    ✅ oro7x 写入完成
  ✅ 汇总对比 Sheet 写入完成

✅ 文件已保存：D:\周报更新版\大盘周报汇总.xlsx


In [3]:
"""
大盘周报生成器 - 多盘合并版 (自动匹配文件路径优化版)
每个盘一个 Sheet + 汇总对比 Sheet

★ 口径说明（与PDF周报统一）：
  充提差率 = sum(充提差) / sum(充值金额) × 100%
  盈亏率   = sum(公司输赢) / sum(投注金额) × 100%
  日均消耗 / ROI：本周固定剔除最后一天（消耗未录完），上周取全7天
"""
import warnings
from pathlib import Path
from datetime import datetime
import numpy as np
import pandas as pd
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

warnings.filterwarnings("ignore")

# ══════════════════════════════════════════════════════════════════════════════
# ★★★ 配置区（每次只改这里，下面无需再动！）★★★
# ══════════════════════════════════════════════════════════════════════════════

# 1. 统计周期配置
THIS_WEEK = ("20260612", "20260618")
LAST_WEEK = ("20260605", "20260611")

# 2. 基础路径配置
DATA_ROOT   = Path(r"D:\周报更新版")       # 周报总根目录
OUTPUT_PATH = DATA_ROOT / "大盘周报汇总.xlsx" # 输出路径自动拼接

# 3. 各平台文件夹结构定义 (只需定义盘名和其子文件夹名称)
# 代码会自动去对应的目录下，寻找最新导出的“平台报表”和“大盘日报”
PLATFORM_CONFIGS = [
    {"name": "MX",      "dir": "MX"},
    {"name": "SOLUNO",  "dir": "SOLUNO"},
    {"name": "SOL777",  "dir": "SOL777"},
    {"name": "Lucro",   "dir": "Lucro"},
    {"name": "wey7",    "dir": "wey7"},
    {"name": "oro7x",   "dir": "oro7x"},
]

# ══════════════════════════════════════════════════════════════════════════════
# 路径自动检索工具函数
# ══════════════════════════════════════════════════════════════════════════════
def find_latest_file(folder_path: Path, pattern: str) -> Path:
    """
    在指定文件夹（及其大盘子文件夹）中，自动模糊匹配符合pattern的文件，并返回最新修改的那一个。
    """
    if not folder_path.exists():
        raise FileNotFoundError(f"找不到平台主目录: {folder_path}")
        
    # 同时检索当前目录以及常出现的“大盘”子目录
    search_paths = [folder_path, folder_path / "大盘"]
    matched_files = []
    
    for path in search_paths:
        if path.exists():
            matched_files.extend(list(path.glob(pattern)))
            
    if not matched_files:
        raise FileNotFoundError(f"在 {folder_path} 中未找到匹配 [{pattern}] 的 Excel 文件")
        
    # 按文件最后修改时间排序，取最新的一个文件（防止重复导出时选错）
    matched_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
    return matched_files[0]

# ══════════════════════════════════════════════════════════════════════════════
# 颜色常量 与 基础工具函数 (保持原样)
# ══════════════════════════════════════════════════════════════════════════════
HDR_BG   = "1E3A5F"; HDR_FG   = "FFFFFF"; MET_BG   = "D6E4F0"; MET_FG   = "1E3A5F"
SUB_HDR  = "2C5F8A"; UP_FG    = "1A7C3E"; DN_FG    = "C0392B"; NEU_FG   = "888888"
ZEBRA    = "F0F6FC"; WHITE    = "FFFFFF"; BORDER_C = "B0C4D8"; COST_BG  = "FFF9E6"

def clean(v):
    try: return float(str(v).replace(",", "").replace("%", "").strip())
    except: return np.nan

def load_and_clean(path):
    df = pd.read_excel(path)
    df["日期"] = df["日期"].astype(str).str.strip()
    for col in df.columns:
        if col != "日期": df[col] = df[col].apply(clean)
    return df

def week_slice(df, week):
    return df[df["日期"].between(week[0], week[1])].sort_values("日期").reset_index(drop=True)

def pct_chg(tw_val, lw_val):
    if (lw_val is not None and not np.isnan(lw_val) and lw_val != 0 and tw_val is not None and not np.isnan(tw_val)):
        return (tw_val - lw_val) / abs(lw_val) * 100
    return np.nan

def safe_sum(series):
    v = series.dropna()
    return float(v.sum()) if len(v) else np.nan

def safe_mean(series):
    v = series.dropna()
    return float(v.mean()) if len(v) else np.nan

def fmt_week_label(week):
    s, e = week
    return f"{s[4:6]}/{s[6:]}—{e[4:6]}/{e[6:]}"

# ══════════════════════════════════════════════════════════════════════════════
# 核心计算与 Sheet 写入 (保持原逻辑不变)
# ══════════════════════════════════════════════════════════════════════════════
def compute_metrics(plat_path, daily_path):
    try:
        p = load_and_clean(plat_path)
        d = load_and_clean(daily_path)
    except Exception as ex:
        return None, f"文件读取失败：{ex}"

    tw_p = week_slice(p, THIS_WEEK); lw_p = week_slice(p, LAST_WEEK)
    tw_d = week_slice(d, THIS_WEEK); lw_d = week_slice(d, LAST_WEEK)

    if len(tw_p) == 0 or len(lw_p) == 0: return None, "数据不足（平台报表无记录）"
    if len(tw_d) == 0 or len(lw_d) == 0: return None, "数据不足（日报无记录）"

    tw_cost_rows = tw_d.iloc[:-1]; lw_cost_rows = lw_d
    tw_cost_days = len(tw_cost_rows); lw_cost_days = len(lw_cost_rows)

    tw_cost_avg = (safe_sum(tw_cost_rows["真实消耗"]) / tw_cost_days if tw_cost_days else np.nan)
    lw_cost_avg = (safe_sum(lw_cost_rows["真实消耗"]) / lw_cost_days if lw_cost_days else np.nan)

    cd_col = "充提差" if "充提差" in tw_d.columns else None
    if cd_col:
        tw_cd_valid = tw_d.iloc[:-1]["充提差"]
        lw_cd_all   = lw_d["充提差"]
    else:
        tw_cd_valid = tw_p.sort_values("日期").iloc[:-1]["充提差"]
        lw_cd_all   = lw_p.sort_values("日期")["充提差"]

    tw_cd_avg = safe_sum(tw_cd_valid) / tw_cost_days if tw_cost_days else np.nan
    lw_cd_avg = safe_sum(lw_cd_all)   / lw_cost_days if lw_cost_days else np.nan

    tw_roi = (tw_cd_avg / tw_cost_avg if tw_cost_avg and not np.isnan(tw_cost_avg) else np.nan)
    lw_roi = (lw_cd_avg / lw_cost_avg if lw_cost_avg and not np.isnan(lw_cost_avg) else np.nan)

    tw_充值  = safe_sum(tw_p["充值金额"]); lw_充值  = safe_sum(lw_p["充值金额"])
    tw_充提差 = safe_sum(tw_p["充提差"]);   lw_充提差 = safe_sum(lw_p["充提差"])
    tw_投注  = safe_sum(tw_p["投注金额"]) if "投注金额" in tw_p.columns else np.nan
    lw_投注  = safe_sum(lw_p["投注金额"]) if "投注金额" in lw_p.columns else np.nan
    tw_输赢  = safe_sum(tw_p["公司输赢"]); lw_输赢  = safe_sum(lw_p["公司输赢"])
    tw_注册  = safe_sum(tw_p["注册人数"]); lw_注册  = safe_sum(lw_p["注册人数"])
    tw_首充  = safe_sum(tw_p["首充人数"]); lw_首充  = safe_sum(lw_p["首充人数"])
    tw_活跃  = safe_sum(tw_p["活跃人数"]); lw_活跃  = safe_sum(lw_p["活跃人数"])

    tw_充提差率 = tw_充提差 / tw_充值 * 100 if tw_充值 else np.nan
    lw_充提差率 = lw_充提差 / lw_充值 * 100 if lw_充值 else np.nan
    tw_盈亏率 = tw_输赢 / tw_投注 * 100 if (tw_投注 and not np.isnan(tw_投注)) else np.nan
    lw_盈亏率 = lw_输赢 / lw_投注 * 100 if (lw_投注 and not np.isnan(lw_投注)) else np.nan

    tw_日均活跃   = tw_活跃 / 7 if tw_活跃 else np.nan
    lw_日均活跃   = lw_活跃 / 7 if lw_活跃 else np.nan
    tw_首充转化率 = tw_首充 / tw_注册 * 100 if tw_注册 else np.nan
    lw_首充转化率 = lw_首充 / lw_注册 * 100 if lw_注册 else np.nan
    tw_首充ARPPU  = safe_mean(tw_p["首充Arppu"]);  lw_首充ARPPU  = safe_mean(lw_p["首充Arppu"])
    tw_老用户ARPPU = safe_mean(tw_p["老用户ARPPU"]); lw_老用户ARPPU = safe_mean(lw_p["老用户ARPPU"])
    tw_全量ARPPU  = safe_mean(tw_p["全量Arppu"]);  lw_全量ARPPU  = safe_mean(lw_p["全量Arppu"])

    def row(label, tw_val, lw_val, fmt, note="", is_cost=False):
        return {"指标": label, "上周": lw_val, "本周": tw_val, "环比": pct_chg(tw_val, lw_val), "fmt": fmt, "note": note, "is_cost": is_cost}

    cost_note = f"本周取前{tw_cost_days}天/上周取{lw_cost_days}天日均"
    rows = [
        row("充值金额（万）",          tw_充值  / 10000, lw_充值  / 10000, "万"),
        row("充提差率",                tw_充提差率,       lw_充提差率,       "%",  "sum(充提差)/sum(充值金额)"),
        row("公司输赢（万）",          tw_输赢  / 10000, lw_输赢  / 10000, "万"),
        row("盈亏率",                  tw_盈亏率,         lw_盈亏率,         "%",  "sum(公司输赢)/sum(投注金额)"),
        row("日均消耗（万）",          tw_cost_avg / 10000, lw_cost_avg / 10000, "万",  cost_note, True),
        row("日均ROI（充提差/消耗）",  tw_roi,            lw_roi,            "x",  cost_note, True),
        row("注册人数",                tw_注册,           lw_注册,           "整数"),
        row("首充人数",                tw_首充,           lw_首充,           "整数"),
        row("日均活跃",                tw_日均活跃,       lw_日均活跃,       "整数"),
        row("首充转化率",              tw_首充转化率,     lw_首充转化率,     "%"),
        row("首充ARPPU",               tw_首充ARPPU,      lw_首充ARPPU,      "小数"),
        row("老用户ARPPU",             tw_老用户ARPPU,    lw_老用户ARPPU,    "小数"),
        row("全量ARPPU",               tw_全量ARPPU,      lw_全量ARPPU,      "小数"),
    ]
    return rows, None

def mk_border(color=BORDER_C, style="thin"):
    s = Side(style=style, color=color)
    return Border(left=s, right=s, top=s, bottom=s)

def mk_fill(hex_color): return PatternFill("solid", fgColor=hex_color)

def fmt_val(v, fmt):
    if v is None or (isinstance(v, float) and np.isnan(v)): return "-"
    if fmt == "万":   return f"{v:,.2f}万"
    if fmt == "%":    return f"{v:.2f}%"
    if fmt == "x":    return f"{v:.4f}x"
    if fmt == "整数": return f"{int(round(v)):,}"
    if fmt == "小数": return f"{v:.2f}"
    return str(v)

def fmt_chg(chg):
    if chg is None or (isinstance(chg, float) and np.isnan(chg)): return "-"
    return f"{'+' if chg >= 0 else ''}{chg:.2f}%"

def write_platform_sheet(ws, platform_name, rows):
    tw_lbl = fmt_week_label(THIS_WEEK); lw_lbl = fmt_week_label(LAST_WEEK)
    for col, w in zip("ABCDE", [26, 18, 18, 13, 34]): ws.column_dimensions[col].width = w

    ws.merge_cells("A1:E1")
    c = ws["A1"]; c.value = f"【{platform_name}】大盘周报    本周 {tw_lbl}  vs  上周 {lw_lbl}"
    c.font = Font(name="Arial", bold=True, size=12, color=HDR_FG); c.fill = mk_fill(HDR_BG)
    c.alignment = Alignment(horizontal="center", vertical="center"); c.border = mk_border(HDR_BG)
    ws.row_dimensions[1].height = 26

    headers = ["指标", f"上周（{lw_lbl}）", f"本周（{tw_lbl}）", "环比", "口径说明"]
    for col_i, hdr in enumerate(headers, 1):
        c = ws.cell(row=2, column=col_i, value=hdr)
        c.font = Font(name="Arial", bold=True, size=10, color=HDR_FG); c.fill = mk_fill(SUB_HDR)
        c.alignment = Alignment(horizontal="center", vertical="center"); c.border = mk_border(SUB_HDR)
    ws.row_dimensions[2].height = 20

    for i, r in enumerate(rows):
        row_num = i + 3
        bg = COST_BG if r["is_cost"] else (WHITE if i % 2 == 0 else ZEBRA)
        chg = r["环比"]; chg_txt = fmt_chg(chg)
        chg_clr = NEU_FG if chg_txt == "-" else (UP_FG if chg >= 0 else DN_FG)

        cells = [
            (1, r["指标"], MET_BG, "left", Font(name="Arial", bold=True, size=10, color=MET_FG)),
            (2, fmt_val(r["上周"], r["fmt"]), bg, "right", Font(name="Arial", size=10, color="555555")),
            (3, fmt_val(r["本周"], r["fmt"]), bg, "right", Font(name="Arial", bold=True, size=10, color=MET_FG)),
            (4, chg_txt, bg, "center", Font(name="Arial", bold=True, size=10, color=chg_clr)),
            (5, r.get("note", ""), bg, "left", Font(name="Arial", size=9, color="888888", italic=True)),
        ]
        for col_i, val, fill_c, align, font in cells:
            c = ws.cell(row=row_num, column=col_i, value=val)
            c.font = font; c.fill = mk_fill(fill_c)
            c.alignment = Alignment(horizontal=align, vertical="center", indent=(1 if align == "left" else 0))
            c.border = mk_border()
        ws.row_dimensions[row_num].height = 20

    ws.cell(row=len(rows) + 3, column=1, value=f"生成时间：{datetime.now().strftime('%Y-%m-%d %H:%M')}").font = Font(name="Arial", size=8, color="AAAAAA", italic=True)

def write_summary_sheet(ws, summary):
    valid = [(n, r) for n, r in summary if r is not None]
    n_plat = len(valid); tw_lbl = fmt_week_label(THIS_WEEK); lw_lbl = fmt_week_label(LAST_WEEK)

    ws.column_dimensions["A"].width = 26
    for i in range(n_plat * 2): ws.column_dimensions[get_column_letter(i + 2)].width = 13
    end_col = get_column_letter(n_plat * 2 + 1)

    ws.merge_cells(f"A1:{end_col}1")
    c = ws["A1"]; c.value = f"各盘大盘周报汇总对比    本周 {tw_lbl}  vs  上周 {lw_lbl}"
    c.font = Font(name="Arial", bold=True, size=12, color=HDR_FG); c.fill = mk_fill(HDR_BG)
    c.alignment = Alignment(horizontal="center", vertical="center"); c.border = mk_border(HDR_BG)
    ws.row_dimensions[1].height = 26

    ws.cell(row=2, column=1, value="指标")
    for s in ["A2", "A2"]: ws[s].font = Font(name="Arial", bold=True, size=10, color=HDR_FG); ws[s].fill = mk_fill(SUB_HDR); ws[s].alignment = Alignment(horizontal="center", vertical="center"); ws[s].border = mk_border(SUB_HDR)

    for j, (name, _) in enumerate(valid):
        cs = j * 2 + 2; ce = cs + 1
        ws.merge_cells(start_row=2, start_column=cs, end_row=2, end_column=ce)
        c = ws.cell(row=2, column=cs, value=name)
        c.font = Font(name="Arial", bold=True, size=10, color=HDR_FG); c.fill = mk_fill(SUB_HDR)
        c.alignment = Alignment(horizontal="center", vertical="center"); c.border = mk_border(SUB_HDR)
    ws.row_dimensions[2].height = 20

    ws.cell(row=3, column=1, value="").fill = mk_fill(SUB_HDR); ws.cell(row=3, column=1).border = mk_border(SUB_HDR)
    for j in range(n_plat):
        cs = j * 2 + 2
        for offset, sub in enumerate(["本周值", "环比"]):
            c = ws.cell(row=3, column=cs + offset, value=sub)
            c.font = Font(name="Arial", bold=True, size=9, color=HDR_FG); c.fill = mk_fill("3A7DBF")
            c.alignment = Alignment(horizontal="center", vertical="center"); c.border = mk_border("3A7DBF")
    ws.row_dimensions[3].height = 18

    first_rows = valid[0][1]
    for i, tmpl in enumerate(first_rows):
        row_num = i + 4; is_cost = tmpl["is_cost"]
        bg = COST_BG if is_cost else (WHITE if i % 2 == 0 else ZEBRA)

        label = tmpl["指标"]; note = tmpl.get("note", ""); display = f"{label}\n({note})" if note and not is_cost else label
        c = ws.cell(row=row_num, column=1, value=display)
        c.font = Font(name="Arial", bold=True, size=9, color=MET_FG); c.fill = mk_fill(MET_BG)
        c.alignment = Alignment(horizontal="left", vertical="center", indent=1, wrap_text=bool(note and not is_cost))
        c.border = mk_border()
        ws.row_dimensions[row_num].height = 30 if note and not is_cost else 20

        for j, (_, plat_rows) in enumerate(valid):
            cs = j * 2 + 2
            if plat_rows is None:
                for offset in range(2):
                    c = ws.cell(row=row_num, column=cs + offset, value="-"); c.fill = mk_fill(bg)
                    c.alignment = Alignment(horizontal="center", vertical="center"); c.border = mk_border()
                continue

            r = plat_rows[i]; val_str = fmt_val(r["本周"], r["fmt"]); chg = r["环比"]; chg_txt = fmt_chg(chg)
            chg_clr = NEU_FG if chg_txt == "-" else (UP_FG if chg >= 0 else DN_FG)

            cv = ws.cell(row=row_num, column=cs, value=val_str)
            cv.font = Font(name="Arial", size=10, color=MET_FG); cv.fill = mk_fill(bg); cv.alignment = Alignment(horizontal="right", vertical="center"); cv.border = mk_border()

            cc = ws.cell(row=row_num, column=cs + 1, value=chg_txt)
            cc.font = Font(name="Arial", bold=True, size=10, color=chg_clr); cc.fill = mk_fill(bg); cc.alignment = Alignment(horizontal="center", vertical="center"); cc.border = mk_border()

    note_row = len(first_rows) + 5
    ws.merge_cells(f"A{note_row}:{end_col}{note_row}")
    c = ws.cell(row=note_row, column=1, value="口径说明：充提差率=sum(充提差)/sum(充值金额)；盈亏率=sum(公司输赢)/sum(投注金额)；日均消耗/ROI本周取前N-1天（剔最后一天消耗未录完），上周取全7天")
    c.font = Font(name="Arial", size=8, color="666666", italic=True); c.alignment = Alignment(horizontal="left", vertical="center", indent=1)
    ws.cell(row=note_row + 1, column=1, value=f"生成时间：{datetime.now().strftime('%Y-%m-%d %H:%M')}").font = Font(name="Arial", size=8, color="AAAAAA", italic=True)


# ══════════════════════════════════════════════════════════════════════════════
# 主流程 (已优化路径查找逻辑)
# ══════════════════════════════════════════════════════════════════════════════
def main():
    wb = openpyxl.Workbook()
    wb.remove(wb.active)

    summary = []

    print(f"🚀 开始自动扫描根目录下的各盘周报数据: {DATA_ROOT}\n")

    for cfg in PLATFORM_CONFIGS:
        name = cfg["name"]
        platform_dir = DATA_ROOT / cfg["dir"]
        print(f"  处理 【{name}】 ...")
        
        # 🟢 自动化核心：模糊匹配并提取最新的报表文件路径
        try:
            plat_path = find_latest_file(platform_dir, "平台报表_USD_*.xlsx")
            daily_path = find_latest_file(platform_dir, "日报-大盘日报_*.xlsx")
            
            print(f"    📂 自动匹配到平台报表: {plat_path.name}")
            print(f"    📂 自动匹配到大盘日报: {daily_path.name}")
            
        except Exception as err_find:
            print(f"    ⚠️  {name} 文件检索失败：{err_find}")
            summary.append((name, None))
            continue
            
        # 开始指标计算与写入
        rows, err = compute_metrics(plat_path, daily_path)
        if err:
            print(f"    ⚠️  {name} 数据加载失败：{err}")
            summary.append((name, None))
            continue
            
        ws = wb.create_sheet(title=name)
        write_platform_sheet(ws, name, rows)
        summary.append((name, rows))
        print(f"    ✅ {name} 写入完成\n")

    valid_count = sum(1 for _, r in summary if r is not None)
    if valid_count >= 1:
        ws_sum = wb.create_sheet(title="汇总对比", index=0)
        write_summary_sheet(ws_sum, summary)
        print("➡️  汇总对比 Sheet 写入完成")

    OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    wb.save(str(OUTPUT_PATH))
    print(f"\n✨ 【大功告成】文件已成功保存至：{OUTPUT_PATH}")


if __name__ == "__main__":
    main()

🚀 开始自动扫描根目录下的各盘周报数据: D:\周报更新版

  处理 【MX】 ...
    📂 自动匹配到平台报表: 平台报表_USD_20260619142818.xlsx
    📂 自动匹配到大盘日报: 日报-大盘日报_20260619.xlsx
    ✅ MX 写入完成

  处理 【SOLUNO】 ...
    📂 自动匹配到平台报表: 平台报表_USD_20260619170834.xlsx
    📂 自动匹配到大盘日报: 日报-大盘日报_20260619(9).xlsx
    ✅ SOLUNO 写入完成

  处理 【SOL777】 ...
    📂 自动匹配到平台报表: 平台报表_USD_20260619171033.xlsx
    📂 自动匹配到大盘日报: 日报-大盘日报_20260619(5).xlsx
    ✅ SOL777 写入完成

  处理 【Lucro】 ...
    📂 自动匹配到平台报表: 平台报表_USD_20260619151840.xlsx
    📂 自动匹配到大盘日报: 日报-大盘日报_20260619(4).xlsx
    ✅ Lucro 写入完成

  处理 【wey7】 ...
    📂 自动匹配到平台报表: 平台报表_USD_20260619171246.xlsx
    📂 自动匹配到大盘日报: 日报-大盘日报_20260619(7).xlsx
    ✅ wey7 写入完成

  处理 【oro7x】 ...
    📂 自动匹配到平台报表: 平台报表_USD_20260619171526.xlsx
    📂 自动匹配到大盘日报: 日报-大盘日报_20260619(1).xlsx
    ✅ oro7x 写入完成

➡️  汇总对比 Sheet 写入完成

✨ 【大功告成】文件已成功保存至：D:\周报更新版\大盘周报汇总.xlsx
